In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

from core.logging import configure_logging
from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from registry import RegistryClient

from gl.models import GLSegments
from gl import GLClient
from gl.repository import GLRepository

from recon import ReconClient

from break_analysis.tools import RegistryTools
from break_analysis import BreakAnalysisAgent
from break_analysis.models import BreakRecord
from break_analysis.builder import BreakCaseBuilder


configure_logging()

def display_df(df):
    display(df.toPandas())

In [3]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('break-agent-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/12 16:48:11 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/09/12 16:48:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7ec0ad5d-6c66-4438-9b59-30d11e16a89f;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 67ms :: artifacts dl 2ms
	:: modules in use:
	org.checkerframework#

In [4]:
registry_client = RegistryClient.from_db(
    spark=spark,
    entity_table='registry.gl_entity',
    department_table='registry.gl_dept',
    branch_table='registry.gl_branch',
    account_table='registry.gl_account',
    sub_account_table='registry.gl_sub_account',
    affiliate_table='registry.gl_affiliate',
    product_table='registry.gl_product',
    book_table='registry.gl_book',
    source_table='registry.gl_source',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

gl_store = PostgresStore(
    spark,
    table_names={
        'SEGMENT_DEFAULT': 'gl.segment_default',
        'POSTING': 'gl.posting',
        'REJECTION': 'gl.rejection',
        'INTERFACE_TRIAL_BALANCE': 'interface.trial_balance',
    },
)

gl_repository = GLRepository(gl_store, spark)
gl = GLClient(gl_repository, registry_client)

recon = ReconClient.from_db(
    spark=spark,
    run_tracker=run_tracker,
    gl=gl,
)

registry_tools = RegistryTools(registry_client=registry_client)

In [5]:
llm = ChatOllama(
    model='qwen3:14b-q4_K_M',
    temperature=0
)

agent = BreakAnalysisAgent(
    llm=llm,
    registry_tools=registry_tools
)

In [6]:
workflow_run_id = '694af9a4-9c03-419f-a9d6-705ab094677d'

recon_df = recon.get_results(workflow_run_id)
breaks_df = recon_df.filter(F.col('DIFFERENCE_AMOUNT') != 0)

breaks = []
for row in breaks_df.collect():
    segments = GLSegments(
        entity_cd = row['ENTITY_CD'],
        branch_cd = row['BRANCH_CD'],
        dept_cd = row['DEPT_CD'],
        gl_account = row['GL_ACCOUNT'],
        sub_account = row['SUB_ACCOUNT'],
        affiliate_cd = row['AFFILIATE_CD'],
        product_cd = row['PRODUCT_CD'],
        book_cd = row['BOOK_CD'],
        source_cd = row['SOURCE_CD'],
    )
    breaks.append(
        BreakRecord(
            recon_result_id= row['RECON_RESULT_ID'],
            workflow_run_id = row['WORKFLOW_RUN_ID'],
            as_of_date = row['AS_OF_DATE'],
            segments = segments,
            accounted_currency = row['ACCOUNTED_CURRENCY'],
            interface_balance = row['INTERFACE_BALANCE'],
            gl_balance = row['GL_BALANCE'],
            difference_amount = row['DIFFERENCE_AMOUNT']
        )
    )

segment_defaults = gl.get_segment_defaults()
case_builder = BreakCaseBuilder(segment_defaults)

break_cases = case_builder.build(breaks)

2026-09-12 16:48:16,352 | INFO | break_analysis.builder | Building break cases | records=7
2026-09-12 16:48:16,353 | INFO | break_analysis.builder | Break cases built | total=3 | one_to_one=2 | many_to_one=1 | ambiguous=0 | interface_only=0 | gl_only=0 | unmatched=0 | duration_ms=1


In [7]:
break_case = break_cases[0]

agent.analyze(break_case)

2026-09-12 16:48:16,357 | INFO | break_analysis.agent | Analyzing break case | case_id=6957bb7c | workflow_run_id=694af9a4 | topology=MANY_TO_ONE | records=3
2026-09-12 16:48:16,357 | INFO | break_analysis.agent | Invoking LLM | case_id=6957bb7c | round=1
2026-09-12 16:48:23,759 | INFO | break_analysis.agent | LLM invoked | case_id=6957bb7c | round=1 | tool_calls=4 | input_tokens=1115 | output_tokens=203 | total_tokens=1318 | prompt_eval_count=1115 | eval_count=203 | load_ms=125 | prompt_eval_ms=418 | eval_ms=6711 | total_ms=7396 | duration_ms=7402
2026-09-12 16:48:23,760 | INFO | break_analysis.agent | Tool round | case_id=6957bb7c | round=1 | tool_calls=4
2026-09-12 16:48:23,982 | INFO | break_analysis.agent | Tool invoked | case_id=6957bb7c | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "310000", "business_dt": "2026-03-31"} | duration_ms=221
2026-09-12 16:48:24,109 | INFO | break_analysis.agent | Tool invoked | case_id=6957bb7c | tool=validate_segmen

BreakAnalysisResult(case_id=UUID('6957bb7c-015b-4180-b4ea-532ad7475c29'), recon_result_ids=(UUID('dbc5342b-1bc1-4ef9-a77c-2f8a358d383e'), UUID('4fe199b3-8537-4650-9860-a969989d8d43'), UUID('bd5598a3-6a7c-40d0-a3bc-42d2ca52f930')), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, explanation="Invalid segments found: gl_account '310000' (inactive), gl_account '4100000' (missing), sub_account '003000' (inactive), and sub_account '004000' (inactive).")

In [8]:
break_case = break_cases[1]

agent.analyze(break_case)

2026-09-12 16:49:53,119 | INFO | break_analysis.agent | Analyzing break case | case_id=41650c04 | workflow_run_id=694af9a4 | topology=ONE_TO_ONE | records=2
2026-09-12 16:49:53,120 | INFO | break_analysis.agent | Invoking LLM | case_id=41650c04 | round=1
2026-09-12 16:49:55,905 | INFO | break_analysis.agent | LLM invoked | case_id=41650c04 | round=1 | tool_calls=1 | input_tokens=859 | output_tokens=50 | total_tokens=909 | prompt_eval_count=859 | eval_count=50 | load_ms=134 | prompt_eval_ms=327 | eval_ms=1994 | total_ms=2784 | duration_ms=2785
2026-09-12 16:49:55,906 | INFO | break_analysis.agent | Tool round | case_id=41650c04 | round=1 | tool_calls=1
2026-09-12 16:49:56,032 | INFO | break_analysis.agent | Tool invoked | case_id=41650c04 | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "210000", "business_dt": "2026-03-31"} | duration_ms=126
2026-09-12 16:49:56,032 | INFO | break_analysis.agent | Invoking LLM | case_id=41650c04 | round=2
2026-09-12 16:49:5

BreakAnalysisResult(case_id=UUID('41650c04-b61f-492e-a66d-5c815d145ae8'), recon_result_ids=(UUID('e0161602-3a56-4b0c-9165-2a3bfb06f733'), UUID('56113b7e-582f-41d9-98ed-8575fceda4ab')), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, explanation="The 'GL_ACCOUNT' segment with value '210000' exists in Registry but is marked as inactive (status='I') for the business date 2026-03-31.")

In [9]:
break_case = break_cases[2]

agent.analyze(break_case)

2026-09-12 16:50:14,224 | INFO | break_analysis.agent | Analyzing break case | case_id=9020f5ec | workflow_run_id=694af9a4 | topology=ONE_TO_ONE | records=2
2026-09-12 16:50:14,224 | INFO | break_analysis.agent | Invoking LLM | case_id=9020f5ec | round=1
2026-09-12 16:50:23,136 | INFO | break_analysis.agent | LLM invoked | case_id=9020f5ec | round=1 | tool_calls=3 | input_tokens=857 | output_tokens=130 | total_tokens=987 | prompt_eval_count=857 | eval_count=130 | load_ms=125 | prompt_eval_ms=484 | eval_ms=8171 | total_ms=8910 | duration_ms=8912
2026-09-12 16:50:23,136 | INFO | break_analysis.agent | Tool round | case_id=9020f5ec | round=1 | tool_calls=3
2026-09-12 16:50:23,251 | INFO | break_analysis.agent | Tool invoked | case_id=9020f5ec | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "", "business_dt": "2026-03-31"} | duration_ms=114
2026-09-12 16:50:23,361 | INFO | break_analysis.agent | Tool invoked | case_id=9020f5ec | tool=validate_segment | args={

BreakAnalysisResult(case_id=UUID('9020f5ec-8855-4d17-a680-cb1f427d0beb'), recon_result_ids=(UUID('6a1980e4-59fd-4d6b-a905-b745b80468f4'), UUID('0a6a3f0c-b9cf-49ba-8875-14bd28cd23af')), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, explanation='Segments GL_ACCOUNT, GL_SUB_ACCOUNT, and GL_PRODUCT had blank values in the investigation record, which are invalid in Registry. These segments are required and must have resolved values to be valid.')

In [10]:
# spark.stop()